# 05 — Visualizations

**SDC Master's Final Project — soccernet-setpiece-vision**

Animated outputs for thesis + viva demo. Produces three-panel GIFs per representative set-piece clip:

| Panel | Source | What it shows |
|---|---|---|
| Left | SoccerNet GSR broadcast frame + YOLO bboxes coloured by KMeans team | detection quality and team assignment |
| Centre | Top-down minimap (mplsoccer pitch) | players + ball in metric coordinates after homography |
| Right | Pitch Control heatmap (Laurie Shaw, recomputed) | attacker control surface, vendor-pinned in nb03 |

All inputs read from `outputs/*.parquet` produced by nb02 + nb03. No re-detection. Frames are read from the SSD on demand.

Outputs land in `outputs/figures/` as `anim_<action>_<clip_id>.gif` (PillowWriter, no ffmpeg needed).

## 0. Setup

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from mplsoccer import Pitch

PROJECT_ROOT = Path.cwd().parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
GSR_ROOT = Path("/Volumes/MPH-ExternalStorage/soccernet-gsr/gamestate-2024")

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

# PC params (must match nb03 exactly)
GRID_NX, GRID_NY = 60, 40
MAX_SPEED = 5.0
REACTION_TIME = 0.7
SIGMA = 0.45
TIME_TO_INTERCEPT_SIGMOID_K = float(np.pi / (np.sqrt(3.0) * SIGMA))

# Team colours
TEAM_COLOURS = {0: "#1f77b4", 1: "#d62728", -1: "#888888"}
BALL_COLOUR = "#000000"

print("OK")

OK


## 1. Pitch Control (vendored, identical to nb03)

In [2]:
def time_to_intercept(player_xy: np.ndarray, target_xy: np.ndarray) -> np.ndarray:
    if len(player_xy) == 0:
        return np.full((1, len(target_xy)), np.inf)
    d = np.linalg.norm(player_xy[:, None, :] - target_xy[None, :, :], axis=2)
    return REACTION_TIME + d / MAX_SPEED


def pitch_control_surface(att_xy: np.ndarray, def_xy: np.ndarray) -> np.ndarray:
    xs = np.linspace(0.0, PITCH_LENGTH_M, GRID_NX)
    ys = np.linspace(0.0, PITCH_WIDTH_M, GRID_NY)
    grid = np.array(np.meshgrid(xs, ys)).reshape(2, -1).T
    if len(att_xy) == 0 or len(def_xy) == 0:
        return np.full((GRID_NY, GRID_NX), 0.5)
    tti_att = time_to_intercept(att_xy, grid).min(axis=0)
    tti_def = time_to_intercept(def_xy, grid).min(axis=0)
    delta = tti_att - tti_def
    p_att = 1.0 / (1.0 + np.exp(TIME_TO_INTERCEPT_SIGMOID_K * delta))
    return p_att.reshape(GRID_NY, GRID_NX)


def split_attack_defend(players_xy: np.ndarray, team_labels: np.ndarray, ball_xy: tuple):
    bx, by = ball_xy
    teams = [t for t in np.unique(team_labels) if t != -1 and not np.isnan(t)]
    if len(teams) < 2:
        return players_xy, np.zeros((0, 2)), teams[0] if teams else None
    min_d = {}
    for t in teams:
        mask = team_labels == t
        if mask.sum() == 0:
            continue
        d = np.linalg.norm(players_xy[mask] - np.array([bx, by]), axis=1)
        min_d[t] = d.min()
    att = min(min_d.keys(), key=lambda k: min_d[k])
    deff = next(t for t in min_d if t != att)
    return players_xy[team_labels == att], players_xy[team_labels == deff], att


print("PC functions ready")

PC functions ready


## 2. Load detections + ball positions

In [3]:
pipe_df = pd.read_parquet(OUTPUTS_DIR / "detections_pipeline.parquet")
gt_df = pd.read_parquet(OUTPUTS_DIR / "detections_gt.parquet")


def parse_ball_position(clip_path: Path, frame_idx: int):
    label_path = clip_path / "Labels-GameState.json"
    if not label_path.is_file():
        return None
    labels = json.load(open(label_path))
    target = f"{frame_idx:06d}.jpg"
    image_id = next((img["image_id"] for img in labels["images"] if img.get("file_name") == target), None)
    if image_id is None:
        return None
    for a in labels["annotations"]:
        if a.get("image_id") != image_id or a.get("category_id") != 4:
            continue
        bp = a.get("bbox_pitch")
        if not bp:
            continue
        x = bp.get("x_bottom_middle")
        y = bp.get("y_bottom_middle")
        if x is None or y is None:
            continue
        return float(x) + PITCH_LENGTH_M / 2, float(y) + PITCH_WIDTH_M / 2
    return None


def parse_bbox_image(clip_path: Path, frame_idx: int):
    """Return list of {'bbox': (x,y,w,h), 'role': str} from GT JSON for overlaying GT."""
    label_path = clip_path / "Labels-GameState.json"
    labels = json.load(open(label_path))
    target = f"{frame_idx:06d}.jpg"
    image_id = next((img["image_id"] for img in labels["images"] if img.get("file_name") == target), None)
    if image_id is None:
        return []
    out = []
    for a in labels["annotations"]:
        if a.get("image_id") != image_id:
            continue
        b = a.get("bbox_image")
        if not b:
            continue
        out.append({"x": b.get("x"), "y": b.get("y"),
                    "w": b.get("w"), "h": b.get("h"),
                    "category": a.get("category_id")})
    return out


print(f"pipeline: {pipe_df.shape}, gt: {gt_df.shape}")
print("clips with detections:", pipe_df['clip_id'].nunique())

pipeline: (4146, 11), gt: (4295, 9)
clips with detections: 20


## 3. Pick representative clips
One per action_class with the most frames in the ±15 window.
Clips confirmed as visually mismatched to their annotated action_class are excluded.

In [ ]:
# Clips whose SoccerNet action_class annotation does not match the visible content.
# SNGS-125: annotated as Corner but shows a mid-game scene.
# SNGS-131: annotated as Direct free-kick but shows a throw-in.
EXCLUDE_CLIPS = {"SNGS-125", "SNGS-131"}

frame_counts = (pipe_df.groupby(["split", "clip_id", "action_class"])["frame_idx"]
                .nunique().reset_index(name="n_frames"))
frame_counts = frame_counts[~frame_counts["clip_id"].isin(EXCLUDE_CLIPS)]
best = (frame_counts.sort_values(["action_class", "n_frames"], ascending=[True, False])
                    .groupby("action_class").head(1))
print(best.to_string(index=False))

## 4. Animation builder

In [5]:
def animate_clip(split: str, clip_id: str, action_class: str, fps: int = 6, out_path: Path | None = None):
    clip_path = GSR_ROOT / split / clip_id
    sub = pipe_df[(pipe_df["split"] == split) & (pipe_df["clip_id"] == clip_id)].copy()
    frames = sorted(sub["frame_idx"].unique())
    if not frames:
        print(f"  no frames for {clip_id}")
        return None

    fig = plt.figure(figsize=(16, 5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])
    ax_bcast = fig.add_subplot(gs[0])
    ax_mini = fig.add_subplot(gs[1])
    ax_pc = fig.add_subplot(gs[2])
    pitch = Pitch(pitch_type="custom", pitch_length=PITCH_LENGTH_M, pitch_width=PITCH_WIDTH_M,
                  line_color="#444", pitch_color="#fafafa")

    def draw(frame_idx: int):
        ax_bcast.clear(); ax_mini.clear(); ax_pc.clear()

        # --- broadcast ---
        img_path = clip_path / "img1" / f"{frame_idx:06d}.jpg"
        if img_path.is_file():
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            ax_bcast.imshow(img)
        ax_bcast.set_xticks([]); ax_bcast.set_yticks([])
        ax_bcast.set_title(f"Broadcast — frame {frame_idx}")

        # GT bboxes (as light grey, just for context) - omitted to keep frame uncluttered

        # --- frame data ---
        f_sub = sub[sub["frame_idx"] == frame_idx]
        ball_xy = parse_ball_position(clip_path, frame_idx)

        # --- minimap ---
        pitch.draw(ax=ax_mini)
        for team_lbl, c in TEAM_COLOURS.items():
            mask = f_sub["team_kmeans"] == team_lbl
            if mask.any():
                ax_mini.scatter(f_sub.loc[mask, "x_m"], f_sub.loc[mask, "y_m"],
                                s=70, c=c, edgecolors="black", linewidths=0.6, zorder=3)
        if ball_xy is not None:
            ax_mini.scatter([ball_xy[0]], [ball_xy[1]], s=80, c=BALL_COLOUR,
                            marker="*", edgecolors="white", linewidths=0.8, zorder=4)
        ax_mini.set_title("Minimap (metric pitch)")

        # --- PC ---
        pitch.draw(ax=ax_pc)
        if ball_xy is not None and not f_sub.empty:
            players = f_sub[["x_m", "y_m"]].to_numpy()
            teams = f_sub["team_kmeans"].to_numpy()
            att, deff, _ = split_attack_defend(players, teams, ball_xy)
            pc = pitch_control_surface(att, deff)
            extent = (0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M)
            ax_pc.imshow(pc, origin="lower", extent=extent, cmap="RdBu_r",
                         vmin=0, vmax=1, alpha=0.65, zorder=2)
            ax_pc.scatter([ball_xy[0]], [ball_xy[1]], s=60, c=BALL_COLOUR,
                          marker="*", edgecolors="white", zorder=5)
        ax_pc.set_title("Pitch Control (attacker p)")

        fig.suptitle(f"{action_class} — {clip_id}", fontsize=12)

    def update(i):
        draw(frames[i])
        return []

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 // fps, blit=False)
    if out_path is None:
        out_path = FIGURES_DIR / f"anim_{action_class.replace(' ', '_').lower()}_{clip_id}.gif"
    anim.save(out_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f"  saved {out_path.name}  ({len(frames)} frames, {fps} fps)")
    return out_path

## 5. Render

In [6]:
saved = []
for _, row in best.iterrows():
    p = animate_clip(row["split"], row["clip_id"], row["action_class"], fps=6)
    if p is not None:
        saved.append(p)

print("\nDone:")
for p in saved:
    print(f"  {p}")

  saved anim_corner_SNGS-125.gif  (16 frames, 6 fps)


  saved anim_direct_free-kick_SNGS-131.gif  (16 frames, 6 fps)

Done:
  /Users/mph/Dev/itzmore-mph/MAIS-projects/final-master-project/soccernet-setpiece-vision/outputs/figures/anim_corner_SNGS-125.gif
  /Users/mph/Dev/itzmore-mph/MAIS-projects/final-master-project/soccernet-setpiece-vision/outputs/figures/anim_direct_free-kick_SNGS-131.gif


## 6. Static still (single-frame three-panel) for thesis figure
Pick the action_position frame of each chosen clip.

In [7]:
def render_still(split: str, clip_id: str, action_class: str, frame_idx: int):
    clip_path = GSR_ROOT / split / clip_id
    sub = pipe_df[(pipe_df["split"] == split) & (pipe_df["clip_id"] == clip_id) &
                  (pipe_df["frame_idx"] == frame_idx)]
    fig = plt.figure(figsize=(16, 5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])
    ax_bcast = fig.add_subplot(gs[0])
    ax_mini = fig.add_subplot(gs[1])
    ax_pc = fig.add_subplot(gs[2])
    pitch = Pitch(pitch_type="custom", pitch_length=PITCH_LENGTH_M, pitch_width=PITCH_WIDTH_M,
                  line_color="#444", pitch_color="#fafafa")

    img_path = clip_path / "img1" / f"{frame_idx:06d}.jpg"
    if img_path.is_file():
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax_bcast.imshow(img)
    ax_bcast.set_xticks([]); ax_bcast.set_yticks([])
    ax_bcast.set_title(f"Broadcast (frame {frame_idx})")

    ball_xy = parse_ball_position(clip_path, frame_idx)
    pitch.draw(ax=ax_mini)
    for team_lbl, c in TEAM_COLOURS.items():
        mask = sub["team_kmeans"] == team_lbl
        if mask.any():
            ax_mini.scatter(sub.loc[mask, "x_m"], sub.loc[mask, "y_m"],
                            s=70, c=c, edgecolors="black", linewidths=0.6, zorder=3)
    if ball_xy is not None:
        ax_mini.scatter([ball_xy[0]], [ball_xy[1]], s=80, c=BALL_COLOUR,
                        marker="*", edgecolors="white", linewidths=0.8, zorder=4)
    ax_mini.set_title("Minimap")

    pitch.draw(ax=ax_pc)
    if ball_xy is not None and not sub.empty:
        players = sub[["x_m", "y_m"]].to_numpy()
        teams = sub["team_kmeans"].to_numpy()
        att, deff, _ = split_attack_defend(players, teams, ball_xy)
        pc = pitch_control_surface(att, deff)
        ax_pc.imshow(pc, origin="lower", extent=(0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M),
                     cmap="RdBu_r", vmin=0, vmax=1, alpha=0.65, zorder=2)
        ax_pc.scatter([ball_xy[0]], [ball_xy[1]], s=60, c=BALL_COLOUR,
                      marker="*", edgecolors="white", zorder=5)
    ax_pc.set_title("Pitch Control")

    fig.suptitle(f"{action_class} — {clip_id} (frame {frame_idx})", fontsize=12)
    out_path = FIGURES_DIR / f"still_{action_class.replace(' ', '_').lower()}_{clip_id}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


for _, row in best.iterrows():
    sub = pipe_df[(pipe_df["split"] == row["split"]) & (pipe_df["clip_id"] == row["clip_id"])]
    centre = int(np.median(sub["frame_idx"].unique()))
    p = render_still(row["split"], row["clip_id"], row["action_class"], centre)
    print(f"  {p.name}")

  still_corner_SNGS-125.png


  still_direct_free-kick_SNGS-131.png


### Output
- `outputs/figures/anim_corner_*.gif` — animated PC + minimap + broadcast for one corner clip.
- `outputs/figures/anim_direct_free-kick_*.gif` — same for one direct FK clip.
- `outputs/figures/still_*.png` — static three-panel for thesis embedding (around the action_position frame).

Limitations vs the SoccerNet teaser image:
- No persistent player IDs (would require a tracker like ByteTrack — future work).
- Action / replay correspondence is a separate SoccerNet task and out of scope here.